<a href="https://colab.research.google.com/github/LegalIntermediaSL/Nautica/blob/main/simulaciones/15_consumo_combustible_autonomia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulación 15: Consumo de Combustible y Autonomía (Millas Náuticas)

Planificar una travesía a motor exige responder una pregunta muy concreta: **¿hasta dónde puedo llegar con el gasoil que llevo?** La autonomía no es una cifra fija: depende de la velocidad a la que navegues, porque el consumo (litros/hora) no crece de forma lineal con la velocidad — al aproximarse al planeo, el motor exige mucho más combustible por cada nudo adicional.

## Fórmulas empleadas

Litros útiles (descontando la reserva de seguridad, normalmente 10-20% para no quedarse sin combustible ni remover los posos del depósito):
$$ L_{útiles} = L_{depósito} \times (1 - \%_{reserva}) $$

Consumo específico por milla navegada, a una velocidad $v$ (nudos) con un consumo horario $C(v)$ (l/h):
$$ c_{milla}(v) = \frac{C(v)}{v} \quad \text{[litros/milla]} $$

Autonomía en millas náuticas a esa velocidad:
$$ A(v) = \frac{L_{útiles}}{c_{milla}(v)} = \frac{L_{útiles} \times v}{C(v)} $$

Como $C(v)$ crece más rápido que $v$ en régimen de planeo, el consumo por milla ($c_{milla}$) suele tener un **mínimo** en una velocidad de crucero "óptima": navegar más despacio consume poco por hora pero tarda mucho (motor funcionando ineficientemente a bajas RPM); navegar muy rápido dispara el consumo por milla. Esa velocidad de mínimo consumo por milla es la que maximiza la autonomía.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- DATOS DEL DEPÓSITO ---
capacidad_deposito = 200.0   # litros
reserva_seguridad = 0.15     # 15% de reserva (nunca usar hasta la última gota)
litros_utiles = capacidad_deposito * (1 - reserva_seguridad)

# --- CURVA DE CONSUMO DEL MOTOR (datos típicos de fábrica, l/h por velocidad) ---
# A bajas velocidades el barco desplaza; a partir de cierto punto entra en planeo
# y el consumo por hora se dispara.
velocidades_ref = np.array([4, 6, 8, 10, 12, 14, 16, 18, 20])       # nudos
consumo_lh_ref = np.array([4, 5, 8, 13, 22, 38, 54, 70, 85])        # litros/hora

# Interpolamos para tener una curva continua de consumo horario C(v)
velocidades = np.linspace(4, 20, 100)
consumo_lh = np.interp(velocidades, velocidades_ref, consumo_lh_ref)

# --- CONSUMO POR MILLA Y AUTONOMÍA ---
consumo_por_milla = consumo_lh / velocidades              # l/milla
autonomia_millas = litros_utiles / consumo_por_milla        # millas náuticas

# Velocidad de máxima autonomía (mínimo consumo por milla)
idx_optimo = np.argmax(autonomia_millas)
v_optima = velocidades[idx_optimo]
autonomia_max = autonomia_millas[idx_optimo]

print("--- CALCULADORA DE AUTONOMÍA ---")
print(f"Depósito: {capacidad_deposito:.0f} L | Reserva: {reserva_seguridad*100:.0f}% | Litros útiles: {litros_utiles:.1f} L")
print("-" * 55)
for v in [6, 10, 14, 18, 20]:
    c = np.interp(v, velocidades_ref, consumo_lh_ref)
    a = litros_utiles / (c / v)
    print(f"  A {v:>2.0f} nudos -> consumo {c:>5.1f} l/h ({c/v:4.2f} l/milla) -> autonomía {a:6.0f} millas")
print("-" * 55)
print(f"Velocidad de crucero más eficiente: {v_optima:.1f} nudos -> autonomía máxima {autonomia_max:.0f} millas")

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5.5))

# Curva de autonomía vs velocidad
ax1.plot(velocidades, autonomia_millas, color="tab:blue", linewidth=2, label="Autonomía (millas)")
ax1.axvline(v_optima, color="tab:blue", linestyle="--", alpha=0.5)
ax1.scatter([v_optima], [autonomia_max], color="tab:blue", zorder=5)
ax1.set_xlabel("Velocidad de crucero (nudos)")
ax1.set_ylabel("Autonomía (millas náuticas)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, alpha=0.3)

# Curva de consumo por milla (forma típica en U) en eje secundario
ax2 = ax1.twinx()
ax2.plot(velocidades, consumo_por_milla, color="tab:red", linewidth=2, linestyle=":", label="Consumo (l/milla)")
ax2.set_ylabel("Consumo específico (litros/milla)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Autonomía y Consumo por Milla en función de la Velocidad de Crucero")
fig.tight_layout()
plt.show()

## Conclusión

La curva de autonomía frente a velocidad **no es una línea recta**: existe una velocidad de crucero que maximiza las millas alcanzables con el combustible disponible (mínimo de la curva de litros/milla). Navegar por encima de esa velocidad "gasta" autonomía a cambio de tiempo; navegar muy por debajo también es ineficiente porque el motor trabaja fuera de su régimen óptimo. Esta calculadora es orientativa: en la práctica hay que sumar el efecto del viento, la corriente, el estado de la mar y el trimado del motor, y siempre reservar un margen extra sobre el resultado teórico.